In [12]:
import torch, pickle
from pathlib import Path
from nanochat.gpt import GPT, GPTConfig
import chess, chess.pgn, chess.engine

In [ ]:
ckpt = torch.load("chess_min_gpu.pt", map_location="cpu")
config = GPTConfig(**ckpt["meta"]["model_config"])
model = GPT(config).eval()
model.load_state_dict(ckpt["model"])
stoi = ckpt["meta"]["tokenizer"]["stoi"]
itos = ckpt["meta"]["tokenizer"]["itos"]
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [58]:
engine = chess.engine.SimpleEngine.popen_uci("fairy-stockfish")
engine.configure({ "UCI_LimitStrength": True, "UCI_Elo": 500})

In [59]:
board = chess.Board()
result = engine.play(board, chess.engine.Limit(time=0.1))
board.san(result.move)



'd3'

In [ ]:
def play_game(model_color):
    board = chess.Board()
    tokens = ["<bos>"]

    while not board.is_game_over():
        if board.turn == model_color:
            # Filter tokens to only include those in vocabulary - figure out how to handle this gracefully
            token_ids = [stoi[t] for t in tokens if t in stoi]
            if not token_ids:
                token_ids = [stoi["<bos>"]]
            x = torch.tensor(token_ids, device=device)[None, :]
            logits = model(x[:, -config.sequence_len:])

            legal_san = {board.san(mv) for mv in board.legal_moves}
            mask = torch.full((len(itos),), float("-inf"), device=device)
            for token, idx in stoi.items():
                if token in legal_san:
                    mask[idx] = logits[0, -1, idx]

            probs = torch.softmax(mask, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            move_san = itos[next_id.item()]

        else:
            result = engine.play(board, chess.engine.Limit(time=0.1))
            move_san = board.san(result.move)

        board.push_san(move_san)
        tokens.append(move_san)

    return board.outcome(), tokens

In [ ]:
def evaluate(num_games=100):
    results = {"win": 0, "loss": 0, "draw": 0}

    for i in range(num_games):
        model_color = chess.WHITE if i % 2 == 0 else chess.BLACK
        outcome, game_tokens = play_game(model_color)
        print(f"Model color: {'WHITE' if model_color else 'BLACK'}")
        print(f"{' '.join(game_tokens[1:])}")

        if outcome.winner is None:
            results["draw"] += 1
        elif outcome.winner == model_color:
            results["win"] += 1
        else:
            results["loss"] += 1

        if (i + 1) % 10 == 0:
            print(f"Game {i+1}/{num_games}: {results}")

    total = sum(results.values())
    win_rate = (results["win"] + 0.5 * results["draw"]) / total
    print(f"\nFinal: {results}")
    print(f"Win rate: {win_rate:.1%}")

    return results

In [66]:
evaluate(num_games=2)

Model color: WHITE
Nf3 d6 g3 Nc6 Bg2 Bf5 O-O h6 d3 e6 b3 Nge7 Nc3 e5 e4 Bc8 a4 Qd7 a5 g5 a6 Nd4 Be3 Nxf3+ Bxf3 c5 Be2 b6 f4 Ng6 f5 h5 fxg6 fxg6 Bc1 Be7 Na4 h4 Bg4 Qd8 Qf3 Bf6 Nc3 Kf7 Na4 Qe7 Bd2 d5 Be1 dxe4 dxe4 hxg3 hxg3 Bxa6 Nxb6 Rab8 Na4 Bxf1 Kxf1 Rhf8 Ke2 Qd6 Bd2 Kg7 Nb6 axb6 Be3 Rfd8 b4 cxb4 Bxb6 Qd2+ Kf1 Rf8 Kg1 Ra8 Qf2 Qc3 Rd1 Ra1 Qc5 Rfa8 Qe7+ Kg8 Qf8+ Kh7 Be3 Bg7 Kh2 Rxd1 Bc5 Rda1 Be3 Qe1 Kg2 R8a2 Qe8 Qh1+ Kf2 Re1 Qc8 Raa1 Qg8+ Kh6 Qh8+ Bxh8 Bxg5+ Kg7 Bh6+ Kf7 Be3 Qxe4 Bf4 Qd4+ Kg2 exf4 Kh2 Qe3 Kg2 Rg1+ Kh2 Qf2+ Kh3 Qg2+ Kh4 Rh1+ Kg5 f3 Kf4 Raf1 Kg5 Rhg1 Kh6 Bg7+ Kh7 Qxc2 Bf5 Qxf5 g4 Qe4 g5 f2
Model color: BLACK
d3 e6 a3 Nf6 Nc3 d5 e3 a6 Bd2 c5 g3 Nbd7 h4 b5 Nge2 Bb7 Rh2 Ne5 Ng1 Neg4 Rg2 Be7 Nf3 O-O Ng5 h6 f3 hxg5 Re2 Nd7 hxg5 Bxg5 Qb1 f5 Bh3 Kh7 e4 Bf4 exd5 Bxd5 gxf4 Rh8 fxg4 fxg4 Bf1 g3 Bh3 e5 Ne4 exf4 c4 Bb7 cxb5 Bd5 b6 Bc6 Be6 Ne5 Nxc5 Bd5 b7 Nf7 bxa8=N Kg6 Bc1 Nd6 b3 Nf5 Bxf5+ Kxf5 Kd2 Be4 Nb6 Bf3 Nbd7 Rg8 Ne5 Bg2 a4 Bf1 Re1 g2 Nf7 g1=Q Re5+ Kf6 Kc2 Bh3 Ng5 Qg3 Ra2 Qf2+ K

{'win': 0, 'loss': 1, 'draw': 1}